# Supplementary Figure — joint decoding and clustering structure

This notebook reproduces the **logic of the original clustering figure**
rather than summarizing generic per-archetype cluster scores.

The figure asks whether the same sparse spatial-MS-AA subsets that carry
stronger decoding information also organize subjects by experimental condition
more strongly than size-matched random archetype subsets.

The original figure logic is retained:

- **Panel A**
  - x = decoding z-score
  - y = clustering-effect z-score
  - color = representational scale / model order K
  - marker = sparse subset size top-m
  - dashed guides at zero identify the upper-right "jointly strong" quadrant

- **Panel B**
  - heatmap of `joint_zsum = decode_z + cluster_effect_z`
  - rows = top-m
  - columns = K

For faithful replication, the plotted subset is:
- spatial MS-AA only
- intact condition
- NMI clustering metric
- top-m in {1, 3, 5, 7, 10, 15, 20, 25}

This is a figure-only cleanup. It loads the old exported joint table when
available; otherwise it rebuilds the joint scores exactly from the notebook-014
summary table without rerunning clustering permutations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FIT_SCOPE = "across"

PRIMARY_CLUSTER_METRIC = "nmi"
FIGURE_CONDITION = "intact"
FIGURE_ANALYSIS = "spatial"

TOP_M_VALUES = [1, 3, 5, 7, 10, 15, 20, 25]

CLUSTER_METRICS = [
    "nmi",
    "ari",
    "purity",
    "nearest_centroid_accuracy",
    "silhouette_true_labels",
]

SPATIAL_DENOMINATOR = 700
TEMPORAL_DENOMINATOR = 300

# Match the original notebook.
NORMALIZE_WITHIN = "condition_metric"
DECODE_WEIGHT = 1.0
CLUSTER_WEIGHT = 1.0

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_joint_decoding_clustering.png"
FIG_PDF = OUTPUT_DIR / "Supplement_joint_decoding_clustering.pdf"
SOURCE_CSV = OUTPUT_DIR / "Supplement_joint_decoding_clustering_source.csv"

DPI = 300

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# ============================================================
# Locate precomputed outputs
# ============================================================

JOINT_NAME = f"joint_decoding_clustering_long_{FIT_SCOPE}.csv"
SUMMARY_NAME = f"spatial_enabled_clustering_validation_summary_{FIT_SCOPE}.csv"

candidate_joint = [
    Path("016_joint_decoding_clustering_score") / JOINT_NAME,
    Path("figures/016_joint_decoding_clustering_score") / JOINT_NAME,
    Path("../figures/016_joint_decoding_clustering_score") / JOINT_NAME,
    Path("../../figures/016_joint_decoding_clustering_score") / JOINT_NAME,
    Path(
        "outputs/notebook_figures/"
        "016_joint_decoding_clustering_score"
    ) / JOINT_NAME,
]

candidate_summary = [
    Path("014_spatial_enabled_top_decoding_clustering_validation") / SUMMARY_NAME,
    Path("figures/014_spatial_enabled_top_decoding_clustering_validation") / SUMMARY_NAME,
    Path("../figures/014_spatial_enabled_top_decoding_clustering_validation") / SUMMARY_NAME,
    Path("../../figures/014_spatial_enabled_top_decoding_clustering_validation") / SUMMARY_NAME,
    Path(
        "outputs/notebook_figures/"
        "014_spatial_enabled_top_decoding_clustering_validation"
    ) / SUMMARY_NAME,
]


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


joint_path = first_existing(candidate_joint)
summary_path = first_existing(candidate_summary)

# Bounded recursive fallback from current directory / one parent.
if joint_path is None:
    for root in [Path.cwd(), Path.cwd().parent]:
        try:
            matches = list(root.glob(f"**/{JOINT_NAME}"))
        except (OSError, PermissionError):
            matches = []
        if matches:
            joint_path = matches[0]
            break

if summary_path is None:
    for root in [Path.cwd(), Path.cwd().parent]:
        try:
            matches = list(root.glob(f"**/{SUMMARY_NAME}"))
        except (OSError, PermissionError):
            matches = []
        if matches:
            summary_path = matches[0]
            break

print("Precomputed joint table:", joint_path)
print("Notebook-014 summary:", summary_path)

In [ ]:
# ============================================================
# Load or rebuild the joint table
# ============================================================

def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)

    if not np.isfinite(sd) or sd < 1e-12:
        return np.zeros_like(x, dtype=float)

    return (x - mu) / sd


def percentile_rank_safe(x):
    return (
        pd.Series(x, dtype=float)
        .rank(pct=True, method="average")
        .to_numpy()
    )


def component_ratio(K, analysis_type):
    denom = (
        SPATIAL_DENOMINATOR
        if analysis_type == "spatial"
        else TEMPORAL_DENOMINATOR
    )
    return float(K) / denom


if joint_path is not None:
    joint_df = pd.read_csv(joint_path)
    print("Loaded precomputed joint table:", joint_path)

else:
    if summary_path is None:
        raise FileNotFoundError(
            "Could not find either the old joint table or the notebook-014 "
            "clustering-validation summary. Expected one of:\n"
            f"  {JOINT_NAME}\n"
            f"  {SUMMARY_NAME}\n\n"
            "Run notebook 014 first, or place its exported summary CSV in the "
            "project tree. The expensive clustering permutations do not need "
            "to be rerun if that summary already exists."
        )

    summary_df = pd.read_csv(summary_path)
    print("Rebuilding joint table from:", summary_path)

    summary_df["analysis_type"] = summary_df["analysis_type"].astype(str)
    summary_df["condition"] = summary_df["condition"].astype(str)
    summary_df["K"] = summary_df["K"].astype(int)
    summary_df["top_m"] = summary_df["top_m"].astype(int)

    if "component_ratio" not in summary_df.columns:
        summary_df["component_ratio"] = summary_df.apply(
            lambda r: component_ratio(
                r["K"],
                r["analysis_type"],
            ),
            axis=1,
        )

    summary_df = summary_df[
        summary_df["top_m"].isin(TOP_M_VALUES)
    ].copy()

    rows = []

    for metric in CLUSTER_METRICS:
        obs_col = f"{metric}_observed"
        rand_col = f"{metric}_random_mean"
        diff_col = f"{metric}_observed_minus_random"
        z_col = f"{metric}_z"
        p_col = f"{metric}_p_high"

        required = [
            obs_col,
            rand_col,
            diff_col,
            z_col,
        ]

        missing = [
            c
            for c in required
            if c not in summary_df.columns
        ]

        if missing:
            print(
                f"Skipping {metric}; missing columns: {missing}"
            )
            continue

        tmp = summary_df.copy()
        tmp["cluster_metric"] = metric
        tmp["cluster_observed"] = tmp[obs_col]
        tmp["cluster_random_mean"] = tmp[rand_col]
        tmp["cluster_observed_minus_random"] = tmp[diff_col]
        tmp["cluster_z_vs_random"] = tmp[z_col]
        tmp["cluster_p_high"] = (
            tmp[p_col]
            if p_col in tmp.columns
            else np.nan
        )

        rows.append(tmp)

    if len(rows) == 0:
        raise ValueError(
            "None of the expected clustering metrics were available."
        )

    joint_df = pd.concat(
        rows,
        ignore_index=True,
    )

    # Match old notebook:
    # normalize within condition + clustering metric,
    # not separately by analysis type.
    if NORMALIZE_WITHIN == "condition_metric":
        group_cols = [
            "condition",
            "cluster_metric",
        ]
    elif NORMALIZE_WITHIN == "condition_metric_analysis":
        group_cols = [
            "condition",
            "cluster_metric",
            "analysis_type",
        ]
    elif NORMALIZE_WITHIN == "global_metric":
        group_cols = [
            "cluster_metric",
        ]
    else:
        raise ValueError(
            "Unknown NORMALIZE_WITHIN option."
        )

    joint_df["decode_z"] = np.nan
    joint_df["cluster_effect_z"] = np.nan
    joint_df["decode_percentile"] = np.nan
    joint_df["cluster_percentile"] = np.nan

    for _, idx in joint_df.groupby(
        group_cols
    ).groups.items():

        idx = list(idx)

        joint_df.loc[
            idx,
            "decode_z",
        ] = zscore_safe(
            joint_df.loc[
                idx,
                "decode_mean",
            ]
        )

        joint_df.loc[
            idx,
            "cluster_effect_z",
        ] = zscore_safe(
            joint_df.loc[
                idx,
                "cluster_observed_minus_random",
            ]
        )

        joint_df.loc[
            idx,
            "decode_percentile",
        ] = percentile_rank_safe(
            joint_df.loc[
                idx,
                "decode_mean",
            ]
        )

        joint_df.loc[
            idx,
            "cluster_percentile",
        ] = percentile_rank_safe(
            joint_df.loc[
                idx,
                "cluster_observed_minus_random",
            ]
        )

    joint_df["joint_zsum"] = (
        DECODE_WEIGHT * joint_df["decode_z"]
        + CLUSTER_WEIGHT * joint_df["cluster_effect_z"]
    )

    joint_df["joint_percentile_geom"] = np.sqrt(
        np.clip(
            joint_df["decode_percentile"],
            0,
            1,
        )
        * np.clip(
            joint_df["cluster_percentile"],
            0,
            1,
        )
    )

print("Joint rows:", len(joint_df))

In [ ]:
# ============================================================
# Freeze the exact figure subset
# ============================================================

plot_df = (
    joint_df[
        (joint_df["cluster_metric"].astype(str) == PRIMARY_CLUSTER_METRIC)
        & (joint_df["condition"].astype(str) == FIGURE_CONDITION)
        & (joint_df["analysis_type"].astype(str) == FIGURE_ANALYSIS)
        & (joint_df["top_m"].astype(int).isin(TOP_M_VALUES))
    ]
    .copy()
)

if len(plot_df) == 0:
    raise ValueError(
        "No rows found for spatial / intact / NMI."
    )

required = {
    "K",
    "top_m",
    "decode_z",
    "cluster_effect_z",
    "joint_zsum",
}

missing = required - set(plot_df.columns)

if missing:
    raise ValueError(
        f"Joint table is missing required columns: {missing}"
    )

plot_df["K"] = plot_df["K"].astype(int)
plot_df["top_m"] = plot_df["top_m"].astype(int)

# Rank K for visually even colors while displaying actual K values.
all_k = sorted(
    plot_df["K"].dropna().unique()
)

k_to_rank = {
    k: i
    for i, k
    in enumerate(all_k)
}

plot_df["K_rank"] = (
    plot_df["K"]
    .map(k_to_rank)
)

# Same marker order as the old notebook.
markers = [
    "o",
    "s",
    "^",
    "D",
    "P",
    "X",
    "v",
    "*",
]

marker_map = {
    m: markers[i]
    for i, m
    in enumerate(sorted(TOP_M_VALUES))
}

# Figure source data
plot_df.to_csv(
    SOURCE_CSV,
    index=False,
)

print("Saved figure source data:", SOURCE_CSV)

display(
    plot_df
    .sort_values(
        "joint_zsum",
        ascending=False,
    )
    .head(12)[
        [
            "K",
            "top_m",
            "decode_mean",
            "decode_z",
            "cluster_observed_minus_random",
            "cluster_z_vs_random",
            "cluster_effect_z",
            "joint_zsum",
        ]
    ]
)

In [ ]:
# ============================================================
# REPLICATED TWO-PANEL CLUSTERING FIGURE
# ============================================================

fig = plt.figure(
    figsize=(9.0, 10.2)
)

gs = fig.add_gridspec(
    2,
    1,
    height_ratios=[1.18, 0.88],
    hspace=0.30,
)

axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[1, 0]
)


# ============================================================
# A — decoding vs clustering effect
# ============================================================

cmap = plt.cm.viridis
norm = plt.Normalize(
    vmin=0,
    vmax=max(
        1,
        len(all_k) - 1,
    ),
)

for top_m in sorted(TOP_M_VALUES):

    cur = plot_df[
        plot_df["top_m"] == top_m
    ].copy()

    if len(cur) == 0:
        continue

    axA.scatter(
        cur["decode_z"],
        cur["cluster_effect_z"],
        c=cur["K_rank"],
        cmap=cmap,
        norm=norm,
        s=88,
        marker=marker_map[top_m],
        alpha=0.88,
        edgecolor="0.30",
        linewidth=0.45,
        label=f"top-{top_m}",
        zorder=3,
    )


# Zero guides
axA.axhline(
    0,
    color="0.60",
    linestyle="--",
    linewidth=1.0,
    zorder=1,
)

axA.axvline(
    0,
    color="0.60",
    linestyle="--",
    linewidth=1.0,
    zorder=1,
)


# Label strongest joint combinations
top_cases = (
    plot_df
    .sort_values(
        "joint_zsum",
        ascending=False,
    )
    .head(6)
)

for _, r in top_cases.iterrows():

    axA.annotate(
        f"K={int(r['K'])}",
        xy=(
            r["decode_z"],
            r["cluster_effect_z"],
        ),
        xytext=(4, 5),
        textcoords="offset points",
        fontsize=8.5,
        ha="left",
        va="bottom",
    )


axA.set_xlabel(
    "Decoding z-score",
    fontsize=12,
)

axA.set_ylabel(
    "Clustering-effect z-score",
    fontsize=12,
)

axA.set_title(
    "Sparse subsets with stronger decoding also show stronger condition structure",
    fontsize=13,
    fontweight="bold",
    pad=10,
)


# Marker legend = top-m
legend = axA.legend(
    title="Subset size",
    frameon=False,
    ncol=2,
    loc="lower right",
    fontsize=9,
    title_fontsize=9.5,
)


# K colorbar with actual K labels
sm = plt.cm.ScalarMappable(
    norm=norm,
    cmap=cmap,
)

cbar = fig.colorbar(
    sm,
    ax=axA,
    fraction=0.035,
    pad=0.025,
)

tick_step = max(
    1,
    len(all_k) // 10,
)

tick_positions = np.arange(
    0,
    len(all_k),
    tick_step,
)

if (
    len(tick_positions) == 0
    or tick_positions[-1] != len(all_k) - 1
):
    tick_positions = np.append(
        tick_positions,
        len(all_k) - 1,
    )

cbar.set_ticks(
    tick_positions
)

cbar.set_ticklabels(
    [
        str(all_k[int(i)])
        for i
        in tick_positions
    ]
)

cbar.set_label(
    "Representational scale (K)",
    fontsize=10.5,
)


# ============================================================
# B — joint decoding + clustering score heatmap
# ============================================================

heat = (
    plot_df
    .pivot_table(
        index="top_m",
        columns="K",
        values="joint_zsum",
        aggfunc="mean",
    )
    .reindex(
        index=sorted(TOP_M_VALUES)
    )
    .reindex(
        columns=all_k
    )
)

arr = heat.to_numpy(
    dtype=float
)

vmax = np.nanmax(
    np.abs(arr)
)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

im = axB.imshow(
    arr,
    aspect="auto",
    cmap="coolwarm",
    vmin=-vmax,
    vmax=vmax,
    interpolation="nearest",
)


# All K labels, matching the old figure.
axB.set_xticks(
    np.arange(
        len(all_k)
    )
)

axB.set_xticklabels(
    [
        str(k)
        for k in all_k
    ],
    rotation=55,
    ha="right",
    fontsize=7.5,
)

axB.set_yticks(
    np.arange(
        len(TOP_M_VALUES)
    )
)

axB.set_yticklabels(
    [
        str(m)
        for m in sorted(TOP_M_VALUES)
    ],
    fontsize=9,
)

axB.set_xlabel(
    "Model order K",
    fontsize=12,
)

axB.set_ylabel(
    "top-m",
    fontsize=12,
)

axB.set_title(
    "Joint decoding + clustering score across K and subset size",
    fontsize=13,
    fontweight="bold",
    pad=9,
)


cb = fig.colorbar(
    im,
    ax=axB,
    fraction=0.035,
    pad=0.025,
)

cb.set_label(
    "Joint z-score sum",
    fontsize=10.5,
)


# ============================================================
# Panel letters
# ============================================================

for ax, letter in [
    (axA, "A"),
    (axB, "B"),
]:

    ax.text(
        -0.07,
        1.04,
        letter,
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        ha="left",
        va="top",
    )


fig.suptitle(
    "Decoding-informative spatial archetype subsets also organize subjects by condition",
    fontsize=15,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.11,
    right=0.93,
    top=0.92,
    bottom=0.08,
)

fig.savefig(
    FIG_PNG,
    dpi=DPI,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

### Interpretation

This figure is intended to reproduce the logic of the original clustering
analysis:

- the upper-right quadrant in Panel A contains sparse subsets that score above
  the broader configuration set on both decoding and condition-related
  clustering effect;
- Panel B shows where those joint effects concentrate across model order and
  subset size.

It should not be interpreted as a generic relationship between arbitrary
individual archetype cluster scores and decoding accuracy. The unit in this
figure is a **top-m archetype subset at a particular model order K**.